# Lane 2 on Colab GPU — fine-tune BanglaBERT cross-encoder + go/no-go check

Fine-tunes `csebuetnlp/banglabert` as a `[CLS] context [SEP] prompt+response [SEP]` cross-encoder
on a leak-free, contrastive-paired training set (BenHalluEval same-context right/wrong answers +
our own gold sample-real rows), then checks it against the SAME repeated-CV harness Track A used —
the only fair comparison, since that harness's CV number (0.606) already matched the real
leaderboard score (0.609) almost exactly.

**Decision rule: Lane 2 only replaces Track A if it clearly beats Track A's 0.683 CV Macro-F1 on
the context subset.** If it doesn't, we keep the current 0.609 submission and move on — nothing here
is submitted automatically.

1. mounts Drive, clones/pulls the repo
2. brings in competition data + the two BenHalluEval source files Lane 2 needs
3. rebuilds the leak-free train/val split (deterministic — same output every time)
4. fine-tunes BanglaBERT (~10-20 min on a T4 for ~2,200 rows / 6 epochs w/ early stopping)
5. scores our 130 real context rows, runs repeated CV — **the go/no-go gate**
6. only if it wins: scores test context rows too, for use in combine_and_predict.py later

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection'
import os
assert os.path.exists(f'{DRIVE_DIR}/data/test set.csv'), 'competition data not found on Drive'

In [ ]:
%cd /content
!git clone https://github.com/FaiyazAwsaf/Bengali-LLM-Hallucination-Detection.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

!mkdir -p data models submissions "{DRIVE_DIR}/cache" "{DRIVE_DIR}/models"
!cp "{DRIVE_DIR}/data/dataset samples.json" "{DRIVE_DIR}/data/test set.csv" "{DRIVE_DIR}/data/sample submission.csv" data/

# cache lives directly on Drive via symlink — every score cache is instantly persisted,
# and this also picks up Track A's existing NLI caches if present
!rm -rf data/cache && ln -s "{DRIVE_DIR}/cache" data/cache
!ls data/cache/

!pip install -q -U transformers accelerate sentencepiece

### Get the two Lane 2 source files onto Drive (one-time)

`data/trainset_context.csv` and `data/banglahallueval_qa_1000.csv`, plus the BenHalluEval repo's
`Hallucination Generated Answers/qa_4000.csv`, are gitignored (not in GitHub) — they need to already
be on your Drive under `{DRIVE_DIR}/data/`. If this cell fails, upload them there first (same folder
as `dataset samples.json`), preserving the BenHalluEval folder structure for `qa_4000.csv`.

In [ ]:
import os

GEN_REL = "external/BanglaHalluEval/Hallucination Generated Answers/qa_4000.csv"
required = ["trainset_context.csv", "banglahallueval_qa_1000.csv", GEN_REL]
for rel in required:
    src = f"{DRIVE_DIR}/data/{rel}"
    assert os.path.exists(src), f"missing on Drive: {src} — upload it under data/ first"

!mkdir -p "data/external/BanglaHalluEval/Hallucination Generated Answers"
!cp "{DRIVE_DIR}/data/trainset_context.csv" data/
!cp "{DRIVE_DIR}/data/banglahallueval_qa_1000.csv" data/
!cp "{DRIVE_DIR}/data/{GEN_REL}" "data/{GEN_REL}"
print('all Lane 2 source files in place')

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable a GPU runtime')

In [ ]:
# rebuild the leak-free train/val split (deterministic — seeded, same output every run)
!cd src && python lane2_dataset.py

In [ ]:
# fine-tune (models/ is NOT on Drive by default here — save explicitly after)
!cd src && python lane2_finetune.py

In [ ]:
# checkpoint the fine-tuned model to Drive immediately (don't lose it to a disconnect)
!mkdir -p "{DRIVE_DIR}/models/lane2_banglabert"
!cp -r models/lane2_banglabert/* "{DRIVE_DIR}/models/lane2_banglabert/"
!ls "{DRIVE_DIR}/models/lane2_banglabert"

In [ ]:
# THE GO/NO-GO GATE: score our 130 real context rows, run repeated CV.
# Compare the printed "context" Macro-F1 against Track A's 0.683 — only proceed to
# scoring the test set if this is CLEARLY higher (more than the printed +/- std apart).
!cd src && python lane2_predict.py cv

**Stop here and read the CV report above before continuing.**

- If `context` Macro-F1 clearly beats 0.683 (Track A's current score): run the next cell to score
  the test set, so `combine_and_predict.py` can use Lane 2 in place of Track A.
- If it doesn't: don't run the next cell. Report the numbers back — this becomes a logged experiment,
  not a replacement for Track A. The 0.609 submission stays as our best.

In [ ]:
# only run this if the gate above passed
!cd src && python lane2_predict.py test | tail -3
!ls -la "{DRIVE_DIR}/cache"